# dl_40_dq_checks

The gate. Runs every expectation in `Files/dq/expectations.py`.

Blocking failures **raise**, which stops the pipeline before the semantic
model refreshes. That is deliberate: a stale report beats a wrong one.
Warnings are recorded and the run continues.

In [ ]:
import sys
sys.path.insert(0, "/lakehouse/default/Files/lib")

LIB = "/lakehouse/default/Files"

sys.path.insert(0, f"{LIB}/dq")

import fabric_common as fc
import dq
from expectations import all_expectations

batch_id = fc.new_batch_id()
suite = all_expectations()
print(f"batch {batch_id}: {len(suite)} expectations")

In [ ]:
results = dq.run_suite(spark, suite, batch_id)
print(dq.summarise(results))

for result in results:
    if result.passed:
        continue
    flag = "BLOCK" if result.blocking else "warn "
    print(f"  [{flag}] {result.expectation:48s} {result.failing_rows:6,d} rows")

In [ ]:
import json


def write_diag(name: str, payload: dict) -> None:
    """Structured diagnostics to Files/_diag/.

    Fabric's job API gives no per-cell detail - a failed notebook reports
    "Failed" and nothing else. Writing what happened to a file the deploy
    scripts can read back is the difference between debugging this and guessing.
    """
    os.makedirs("/lakehouse/default/Files/_diag", exist_ok=True)
    path = f"/lakehouse/default/Files/_diag/{name}.json"
    with open(path, "w", encoding="utf-8") as handle:
        json.dump(payload, handle, indent=2, default=str)
    print(f"diagnostics -> {path}")

write_diag("dq_checks", {
    "batch_id": batch_id,
    "summary": dq.summarise(results),
    "results": [
        {
            "expectation": r.expectation,
            "table": r.table,
            "severity": r.severity,
            "failing_rows": r.failing_rows,
            "passed": r.passed,
        }
        for r in results
    ],
})

# Raise LAST, so the diagnostics are always written even on a blocking failure.
dq.assert_no_blocking(results)
print("\nno blocking failures - safe to publish")